In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

bronze_adls_path = "abfss://bronze@"+storage_account+".dfs.core.windows.net/insurance_agents/"
silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/insurance_agents/"

In [0]:
from pyspark.sql.types import *

agents_schema = StructType([

    StructField("agent_id", LongType(), True),
    StructField("agent_name", StringType(), True),
    StructField("license_number", StringType(), True),

    StructField("region", StringType(), True),
    StructField("branch_code", StringType(), True),

    StructField("joining_date", TimestampType(), True),

    StructField("commission_percent", DoubleType(), True),
    StructField("performance_score", DoubleType(), True),

    StructField("active_flag", BooleanType(), True),

    StructField("manager_id", LongType(), True),

    StructField("created_at", TimestampType(), True),
])

In [0]:
agents_df = spark.read \
.option("recursiveFileLookup","true") \
.schema(agents_schema) \
.parquet(bronze_adls_path)

In [0]:
from pyspark.sql.functions import *

agents_df_silver = (
    agents_df
    
    # clean text
    .withColumn("agent_name", trim(col("agent_name")))
    .withColumn("region", trim(col("region")))
    .withColumn("branch_code", trim(col("branch_code")))
    
    # tenure
    .withColumn(
        "agent_tenure_years",
        floor(datediff(current_date(), col("joining_date")) / 365.25)
    )
    
    # performance band
    .withColumn(
        "performance_band",
        when(col("performance_score") >= 4.5, "Excellent")
        .when(col("performance_score") >= 3.5, "Good")
        .when(col("performance_score") >= 2.5, "Average")
        .otherwise("Low")
    )
    
    # agent role from hierarchy
    .withColumn(
        "agent_role",
        when(col("manager_id").isNull(), "Regional Head")
        .when(col("agent_id") <= 30, "Branch Manager")
        .otherwise("Field Agent")
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy(col("agent_id")).orderBy(col("created_at").desc())

agents_df_silver = (
    agents_df_silver
    .withColumn("rnk", rank().over(window))
    .filter(col("rnk")==1)
    .drop(col("rnk"))
)

In [0]:
agents_df_silver_final = agents_df_silver.select(
    # identifiers
    "agent_id",
    "agent_name",
    "license_number",
    # hierarchy
    "agent_role",
    "manager_id",
    # location
    "region",
    "branch_code",
    # performance
    "commission_percent",
    "performance_score",
    "performance_band",
    # tenure
    "joining_date",
    "agent_tenure_years",
    # status
    "active_flag",
    # metadata
    "created_at",
)

agents_df_silver_final.write \
.format("delta") \
.mode("overwrite") \
.save(silver_adls_path)